In [ ]:
# accessing google drive
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/

In [ ]:
!pip install transformers
!pip install happytransformer
!pip install simpletransformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 45.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 28.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 79.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.1/258.1 kB 25.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 69.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 21.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.0/190.0 kB 19.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# importing BERT and GPT models
import transformers, happytransformer, simpletransformers, re
from transformers import TFGPT2LMHeadModel, GPT2Tokenizer
from happytransformer import HappyGeneration, GENSettings, HappyTextToText, TTSettings

In [ ]:
# initiating GPT models
gpt_neo = HappyGeneration(model_type='GPT-NEO', model_name='EleutherAI/gpt-neo-125M')
happy_tt = HappyTextToText('T5',  'prithivida/grammar_error_correcter_v1')
tokenizer = GPT2Tokenizer.from_pretrained("ashiqabdulkhader/GPT2-Poet")
model = TFGPT2LMHeadModel.from_pretrained("ashiqabdulkhader/GPT2-Poet")

In [ ]:
# accepting the number of words and initializing the gpt model on that basis
ip = input('Title: ')
max = int(input('Word Limit: '))
while(True):
  choice = input('Type "s" for short story "p" for poem: ')
  if choice == 's' or choice == 'S':
    criteria = input('Genre: ')
    break
  elif choice == 'p' or choice == 'P':
    criteria = input('Form: ')
    break
  else:
    print('Please type either "p" or "s"...')
gen_sett_args = GENSettings(do_sample=True, top_k=50, max_length=max, min_length=max-50, temperature = 1)
tt_sett_args = TTSettings(do_sample=True, top_k=10,  min_length=1, max_length=100, temperature = 1)
input_ids = tokenizer.encode(ip, return_tensors='tf')

In [ ]:
import pickle
import time

# load the model from disk
genre_model = pickle.load(open('model_saved_stories', 'rb'))
form_model = pickle.load(open('model_saved_poems', 'rb'))

In [ ]:
def generate_story():
  # generation of sentences through gpt
  result = gpt_neo.generate_text(ip, args=gen_sett_args)
  op = ip+result.text

  # checking for and correction of grammar errors if present in the generated string using gramformer
  short_story = ''
  for line in op.split('.'):
    line = 'gec: ' + line + '.'
    gramatically_correct_result = happy_tt.generate_text(line, args=tt_sett_args)
    short_story += ' ' + gramatically_correct_result.text
  return short_story

In [ ]:
genre_list = ['Adventure', 'Comedy', 'Drama', 'Dystopian', 'Fantasy', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Tragedy']
short_story = ''
start_time = time.perf_counter()
while(True):
  short_story = generate_story()
  pred = genre_model.predict([short_story])[0][0]
  end_time = time.perf_counter()
  elapsed_time = end_time - start_time
  print("Elapsed time: ", int(elapsed_time))
  if(genre_list[pred] == criteria or elapsed_time > 300):
    break

In [ ]:
short_story

In [ ]:
def generate_poem():
  # generation of poem through gpt
  result = model.generate(input_ids, do_sample=True, max_length=max, top_k=0, top_p=0.9, temperature=1, num_return_sequences=3)
  poem = tokenizer.decode(result[0], skip_special_tokens=True)
  return poem

In [ ]:
form_list = ['Acrostic', 'Ballad', 'Couplet', 'Elegy', 'Haiku', 'Hymn', 'Limerick', 'Ode', 'Sonnet', 'Villanelle']
poem = ''
start_time = time.perf_counter()
while(True):
  poem = generate_poem()
  pred = form_model.predict([poem])[0][0]
  end_time = time.perf_counter()
  elapsed_time = end_time - start_time
  print("Elapsed time: ", int(elapsed_time))
  if(form_list[pred] == criteria or elapsed_time > 300):
    break

In [ ]:
print(poem)